# LSTM Training — BTC/USDT 1H
Bidirectional LSTM with attention mechanism for BTC price direction prediction.
See `notebooks/CLAUDE.md` for environment setup instructions.

In [ ]:
!pip install -q torch pandas numpy scikit-learn yfinance

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# ── Config ───────────────────────────────────────────────────────────────────
SYMBOL    = 'BTC-USD'
SEQ_LEN   = 60
EPOCHS    = 50
HIDDEN    = 128
NUM_LAYERS = 2
DROPOUT   = 0.2
LR        = 1e-3
BATCH_SIZE = 64
PATIENCE  = 7      # early stopping patience
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {DEVICE}')
print(f'Config: symbol={SYMBOL}, seq_len={SEQ_LEN}, epochs={EPOCHS}, hidden={HIDDEN}')

In [ ]:
import yfinance as yf

df = yf.download(SYMBOL, start='2022-01-01', interval='1h', progress=False)
df = df.dropna()
print(f'Downloaded {len(df)} hourly bars for {SYMBOL}')
df.tail(3)

In [ ]:
def compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()
    rs = avg_gain / (avg_loss + 1e-9)
    return 100 - (100 / (1 + rs))

close = df['Close'].squeeze()
high  = df['High'].squeeze()
low   = df['Low'].squeeze()
vol   = df['Volume'].squeeze()

# Returns
df['ret_1h']  = close.pct_change()
df['ret_4h']  = close.pct_change(4)
df['ret_24h'] = close.pct_change(24)

# RSI
df['rsi_14'] = compute_rsi(close, 14)

# MACD
ema12 = close.ewm(span=12, adjust=False).mean()
ema26 = close.ewm(span=26, adjust=False).mean()
df['macd']        = ema12 - ema26
df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
df['macd_hist']   = df['macd'] - df['macd_signal']

# Bollinger Band width
bb_mid = close.rolling(20).mean()
bb_std = close.rolling(20).std()
df['bb_width'] = (bb_std * 2) / (bb_mid + 1e-9)

# ATR
tr = pd.concat([
    high - low,
    (high - close.shift()).abs(),
    (low  - close.shift()).abs(),
], axis=1).max(axis=1)
df['atr_14'] = tr.rolling(14).mean() / (close + 1e-9)

# Volume ratio
df['vol_ratio'] = vol / (vol.rolling(24).mean() + 1e-9)

# Target: next-bar direction (1 = up, 0 = down)
df['target'] = (close.shift(-1) > close).astype(int)

FEATURES = ['ret_1h', 'ret_4h', 'ret_24h', 'rsi_14', 'macd', 'macd_signal',
            'macd_hist', 'bb_width', 'atr_14', 'vol_ratio']

df = df[FEATURES + ['target']].dropna()
print(f'Feature-engineered dataframe: {df.shape}')
df[FEATURES].describe().round(4)

In [ ]:
class BTCDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray, seq_len: int):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self) -> int:
        return len(self.X) - self.seq_len

    def __getitem__(self, idx: int):
        x_seq = self.X[idx : idx + self.seq_len]
        label  = self.y[idx + self.seq_len]
        return x_seq, label


# Train / val / test split
n = len(df)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

scaler = StandardScaler()
X_all = scaler.fit_transform(df[FEATURES].values)
y_all = df['target'].values

X_train, y_train = X_all[:train_end],    y_all[:train_end]
X_val,   y_val   = X_all[train_end:val_end], y_all[train_end:val_end]
X_test,  y_test  = X_all[val_end:],     y_all[val_end:]

train_ds = BTCDataset(X_train, y_train, SEQ_LEN)
val_ds   = BTCDataset(X_val,   y_val,   SEQ_LEN)
test_ds  = BTCDataset(X_test,  y_test,  SEQ_LEN)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train sequences: {len(train_ds)}  |  Val: {len(val_ds)}  |  Test: {len(test_ds)}')

In [ ]:
class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, 1)

    def forward(self, lstm_out: torch.Tensor) -> torch.Tensor:
        # lstm_out: (batch, seq_len, hidden*2)
        scores  = self.attn(lstm_out).squeeze(-1)           # (batch, seq_len)
        weights = torch.softmax(scores, dim=-1).unsqueeze(-1)  # (batch, seq_len, 1)
        context = (lstm_out * weights).sum(dim=1)           # (batch, hidden*2)
        return context


class BiLSTMAttention(nn.Module):
    def __init__(self, input_size: int, hidden: int, num_layers: int, dropout: float):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.attention = AttentionLayer(hidden)
        self.norm = nn.LayerNorm(hidden * 2)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)
        ctx    = self.attention(out)
        ctx    = self.norm(self.dropout(ctx))
        return self.fc(ctx)


model = BiLSTMAttention(
    input_size=len(FEATURES),
    hidden=HIDDEN,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params:,}')
print(model)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

best_val_loss = float('inf')
patience_count = 0
history = []

for epoch in range(1, EPOCHS + 1):
    # ── Train ────────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss   = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_dl)

    # ── Validate ─────────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            val_loss += criterion(model(xb), yb).item()
    val_loss /= max(len(val_dl), 1)

    scheduler.step(val_loss)
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_count = 0
        torch.save(model.state_dict(), '/tmp/lstm_btc_best.pt')
    else:
        patience_count += 1

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | patience={patience_count}')

    if patience_count >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'\nBest val loss: {best_val_loss:.4f}')

In [ ]:
# Load best checkpoint for evaluation
model.load_state_dict(torch.load('/tmp/lstm_btc_best.pt', map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        xb = xb.to(DEVICE)
        preds = model(xb).argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(yb.numpy())

acc = accuracy_score(all_labels, all_preds)
print(f'Test accuracy: {acc:.4f} ({acc*100:.2f}%)')

# Simple directional Sharpe on test set
test_close = close.iloc[val_end + SEQ_LEN:].values
n_preds = min(len(all_preds), len(test_close) - 1)
if n_preds > 0:
    preds_arr  = np.array(all_preds[:n_preds])
    rets       = np.diff(test_close[:n_preds + 1]) / (test_close[:n_preds] + 1e-9)
    strat_rets = rets * (preds_arr * 2 - 1)   # +1 if long, -1 if short
    sharpe     = (strat_rets.mean() / (strat_rets.std() + 1e-9)) * np.sqrt(24 * 365)
    print(f'Annualised Sharpe (test): {sharpe:.3f}')

In [ ]:
import os

save_path = 'lstm_btc_1h.pt'
torch.save(model.state_dict(), save_path)
print(f'Model saved to {save_path}')
print(f'File size: {os.path.getsize(save_path) / 1024:.1f} KB')